## Importing Libraries

In [21]:
import pandas as pd #Import necessary libraries 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

In [22]:
import warnings #Ignore warnings
warnings.filterwarnings('ignore')

## Import Dataset

In [23]:
data="merged_rows.csv" #Read data into a dataframe
df=pd.read_csv(data, header=None)

## Exploratory Data Analysis (EDA)

In [24]:
df.shape

(157, 11)

In [25]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,DateOrderwasplaced,OrderID,ProductID,QuantityOrdered,TotalRetailPriceforThisOrder,CostPricePerUnit,QUANTITYORDERED,PRICEEACH,SALES,ORDERDATE,STATUS
1,01-Jan-17,123002578,220101400106,2,92.6,20.7,2,46.3,92.6,01-Jan-17,Shipped
2,05-Jan-17,123002579,220101400107,4,184.0,35.0,4,46.0,184.0,05-Jan-17,Shipped
3,12-Jan-17,123002580,220101400108,1,48.0,22.0,1,48.0,48.0,12-Jan-17,Cancelled
4,18-Jan-17,123002581,220101400109,3,150.0,31.0,3,50.0,150.0,18-Jan-17,Shipped


In [27]:
col_names=['DateOrderwasplaced','OrderID','ProductID','QuantityOrdered','TotalRetailPriceforThisOrder','CostPricePerUnit','QUANTITYORDERED','PRICEEACH','SALES','ORDERDATE','STATUS'] #Rename column names 
df.columns=col_names
df.columns

Index(['DateOrderwasplaced', 'OrderID', 'ProductID', 'QuantityOrdered',
       'TotalRetailPriceforThisOrder', 'CostPricePerUnit', 'QUANTITYORDERED',
       'PRICEEACH', 'SALES', 'ORDERDATE', 'STATUS'],
      dtype='object')

In [28]:
df = df.drop(index=0).reset_index(drop=True) #dropping unnecessary 0th cell
df.head()

,DateOrderwasplaced,OrderID,ProductID,QuantityOrdered,TotalRetailPriceforThisOrder,CostPricePerUnit,QUANTITYORDERED,PRICEEACH,SALES,ORDERDATE,STATUS
0,01-Jan-17,123002578,220101400106,2,92.6,20.7,2,46.3,92.6,01-Jan-17,Shipped
1,05-Jan-17,123002579,220101400107,4,184.0,35.0,4,46.0,184.0,05-Jan-17,Shipped
2,12-Jan-17,123002580,220101400108,1,48.0,22.0,1,48.0,48.0,12-Jan-17,Cancelled
3,18-Jan-17,123002581,220101400109,3,150.0,31.0,3,50.0,150.0,18-Jan-17,Shipped
4,24-Jan-17,123002582,220101400110,5,275.0,40.0,5,55.0,275.0,24-Jan-17,Shipped


In [32]:
import pandas as pd

df = pd.read_csv("merged_rows.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Parse date
df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], format="%d-%b-%y", errors="coerce")

# Make sure numeric columns are numeric
num_cols = ["SALES", "QUANTITYORDERED", "QuantityOrdered", "CostPricePerUnit", "PRICEEACH", "TotalRetailPriceforThisOrder"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Monthly sales trend
monthly_sales = (
    df.groupby(df["ORDERDATE"].dt.to_period("M"))
      .agg(total_sales=("SALES", "sum"),
           total_orders=("OrderID", "count"),
           total_quantity=("QUANTITYORDERED", "sum"))
      .reset_index()
)
monthly_sales["ORDERDATE"] = monthly_sales["ORDERDATE"].astype(str)

# Top-selling products
top_products = (
    df.groupby("ProductID")
      .agg(total_quantity=("QUANTITYORDERED", "sum"),
           total_sales=("SALES", "sum"))
      .reset_index()
      .sort_values(["total_quantity", "total_sales"], ascending=False)
)

# Profit margin analysis
if "EstimatedCost" not in df.columns:
    df["EstimatedCost"] = df["CostPricePerUnit"] * df["QUANTITYORDERED"]

df["EstimatedProfit"] = df["SALES"] - df["EstimatedCost"]
df["ProfitMargin"] = df["EstimatedProfit"] / df["SALES"]

profit_summary = pd.DataFrame({
    "metric": ["avg_profit_margin", "min_profit_margin", "max_profit_margin", "avg_estimated_profit"],
    "value": [
        df["ProfitMargin"].mean(),
        df["ProfitMargin"].min(),
        df["ProfitMargin"].max(),
        df["EstimatedProfit"].mean()
    ]
})

print("Monthly sales trend:")
print(monthly_sales)

print("\nTop-selling products:")
print(top_products.head(10))

print("\nProfit margin analysis:")
print(profit_summary)

print("\nRegion-wise revenue:")
print("Not available — no region/state/city/country column in the dataset.")

Monthly sales trend:
   ORDERDATE  total_sales  total_orders  total_quantity
0    2017-01        749.6             5              15
1    2017-02        861.0             5              16
2    2017-03       1032.0             5              18
3    2017-04        901.0             5              15
4    2017-05       1047.0             5              16
5    2017-06       1018.0             5              15
6    2017-07       1295.0             5              18
7    2017-08       1217.0             5              16
8    2017-09       1228.0             5              15
9    2017-10       1584.0             5              18
10   2017-11       1535.0             5              16
11   2017-12       3099.0            10              32
12   2018-01       3610.0            10              34
13   2018-02       3137.0             9              28
14   2018-03       4441.0            11              36
15   2018-04       4562.0            10              33
16   2018-05       5011.0  